# [5.6] Multimodal, Embedding, and Function-Calling Models - Exercises

Implement the small validation tools behind embedding retrieval, representation probes, and function-calling checks before trusting released specialist-model output.

```yaml
gt_tier: GT-1 with GT-0 controlled metric exercises
exercise_id: 5.6-multimodal-embedding-function-models
expected_runtime: 45-75 minutes for CPU exercises; several minutes for CUDA embedding and FunctionGemma preflights
requires_gpu: true for the released-checkpoint preflight; false for the implementation exercises
```

Reading map: review masked mean pooling, cosine retrieval metrics, nearest-centroid probes, and function-calling tool schemas. Failure modes to watch for: padding leakage, unnormalized dot-product retrieval, training-only probe metrics, masking after argmax, and treating schema attribution as causal proof.


In [ ]:
import re
import sys
from dataclasses import dataclass
from pathlib import Path

import torch as t

chapter = "chapter5_modern_architectures"
section = "part6_multimodal_embedding_function_models"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part6_multimodal_embedding_function_models.tests as tests


@dataclass(frozen=True)
class EmbeddingRetrievalReport:
    top1_accuracy: float
    mean_reciprocal_rank: float
    mean_positive_similarity: float
    mean_hard_negative_similarity: float
    mean_margin: float


@dataclass(frozen=True)
class CentroidProbe:
    labels: t.Tensor
    centroids: t.Tensor


@dataclass(frozen=True)
class FunctionCallReport:
    accuracy: float
    tool_accuracy: float
    abstention_accuracy: float
    hallucination_rate: float


@dataclass(frozen=True)
class ParsedFunctionCall:
    name: str | None
    arguments: dict[str, str]


## Masked Mean Pooling

Difficulty: easy. Importance: high. Expected output: the padded sentinel vector should not affect the pooled embedding, so the test expects `[[2.0, 1.0], [4.0, 6.0]]`. Common bug: dividing by `seq_len` instead of the number of unpadded tokens.


In [ ]:
def mean_pool_embeddings(token_embeddings: t.Tensor, attention_mask: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


tests.test_mean_pool_embeddings_ignores_padding_and_matches_reference(mean_pool_embeddings)


## Retrieval Metrics

Difficulty: medium. Importance: high. Expected output: the controlled hard-negative case has ranks `[1, 2, 1]` and top-1 accuracy `2/3`. Common bug: using raw dot products, which lets vector norms dominate semantic similarity.


In [ ]:
def l2_normalize(x: t.Tensor, *, eps: float = 1e-12) -> t.Tensor:
    raise NotImplementedError()


def cosine_similarity_matrix(
    query_embeddings: t.Tensor,
    candidate_embeddings: t.Tensor,
) -> t.Tensor:
    raise NotImplementedError()


def retrieval_ranks(similarity: t.Tensor, target_indices: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


def embedding_retrieval_report(
    query_embeddings: t.Tensor,
    candidate_embeddings: t.Tensor,
    target_indices: t.Tensor,
) -> EmbeddingRetrievalReport:
    raise NotImplementedError()


tests.test_retrieval_metrics_rank_pairs_and_hard_negative_margin(
    cosine_similarity_matrix,
    retrieval_ranks,
    embedding_retrieval_report,
)


## Centroid Probe

Difficulty: medium. Importance: high. Expected output: held-out points from three synthetic clusters should be classified with accuracy `1.0`. Common bug: reporting probe training accuracy instead of held-out accuracy.


In [ ]:
def fit_centroid_probe(embeddings: t.Tensor, labels: t.Tensor) -> CentroidProbe:
    raise NotImplementedError()


def predict_centroid_probe(embeddings: t.Tensor, probe: CentroidProbe) -> t.Tensor:
    raise NotImplementedError()


def centroid_probe_accuracy(embeddings: t.Tensor, labels: t.Tensor, probe: CentroidProbe) -> float:
    raise NotImplementedError()


tests.test_centroid_probe_recovers_heldout_clusters(
    fit_centroid_probe,
    predict_centroid_probe,
    centroid_probe_accuracy,
)


## Function-Call Validity

Difficulty: medium. Importance: high. Expected output: invalid high-logit tools are masked out, no-call examples expose a hallucination rate, and FunctionGemma-style call text is parsed into a function name plus arguments. Common bug: evaluating tool accuracy without no-call labels.


In [ ]:
def mask_disallowed_tools(logits: t.Tensor, allowed_tools: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


def function_call_report(
    logits: t.Tensor,
    labels: t.Tensor,
    *,
    no_call_id: int,
) -> FunctionCallReport:
    raise NotImplementedError()


_FUNCTION_CALL_RE = re.compile(r"call:([A-Za-z0-9_]+)\{([^}]*)\}")
_FUNCTION_ARG_RE = re.compile(r"([A-Za-z0-9_]+):(?:<escape>(.*?)<escape>|([^,{}]+))")


def parse_function_call_text(text: str) -> ParsedFunctionCall:
    raise NotImplementedError()


tests.test_mask_disallowed_tools_blocks_invalid_logits(mask_disallowed_tools)
tests.test_function_call_report_separates_tool_and_abstention_errors(function_call_report)
tests.test_parse_function_call_text_extracts_name_and_arguments(parse_function_call_text)


## Schema-Token Attribution

Difficulty: medium. Importance: medium. Expected output: attribution is exactly `hidden_states @ schema_vectors.T`. Common bug: applying a softmax and treating the result as causal attribution.


In [ ]:
def schema_token_attribution(hidden_states: t.Tensor, schema_vectors: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


tests.test_schema_token_attribution_matches_dot_products(schema_token_attribution)


## Final Verification

After the implementation cells pass, compare your local functions against the reference implementation and then run the released-checkpoint preflight from a Python process with CUDA available:

```python
from part6_multimodal_embedding_function_models import solutions
solutions.run_gpu_test(max_vram_gb=24.0)
```

The current checked path validates pinned public BGE retrieval, direct authenticated EmbeddingGemma retrieval, a pinned public FunctionGemma Mobile Actions checkpoint, and direct authenticated base FunctionGemma CUDA loading.


## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
